# Topic Modeling on `body_anonimizado` — merged_df1

Pipeline:
1. Reconstruct **merged_df1** (conversations + tipificaciones LEFT JOIN)
2. Merge with the anonymized text (`body_anonimizado`)
3. Group messages into conversations by `OpenchannelInteractionId`
4. **LDA** topic modeling on anonymized messages
5. **BERT + LDA ensemble** risk scoring per message
6. Conversation-level topic & risk aggregation
7. Visualizations

---
**Files expected** (same names used across the project):
- `30.01.25 Uic.xlsx` — main export (sheet: `Export`)
- `20.02.25TipificacionesConversaciones.xlsx` — tipifications
- `MOSTRA_1_anonimizado.xlsx` — anonymized output with `body_anonimizado` column

## 0. Imports

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Source Files and Build merged_df1

`merged_df1` = LEFT JOIN of `df_principal` × `df_tipificaciones`  
on `OpenchannelInteractionId` ↔ `id` and `ContactId`.

In [ ]:
# ------------------------------------------------------------------
# Adjust file paths if running on Google Colab:
#   '/content/30.01.25 Uic.xlsx'
#   '/content/20.02.25TipificacionesConversaciones.xlsx'
# ------------------------------------------------------------------

PATH_PRINCIPAL      = '30.01.25 Uic.xlsx'
PATH_TIPIFICACIONES = '20.02.25TipificacionesConversaciones.xlsx'
PATH_ANONIMIZADO    = 'MOSTRA_1_anonimizado.xlsx'

df_principal = pd.read_excel(PATH_PRINCIPAL, sheet_name='Export')
df_tipificaciones = pd.read_excel(PATH_TIPIFICACIONES)

print(f'df_principal      : {df_principal.shape}  columns: {df_principal.columns.tolist()}')
print(f'df_tipificaciones : {df_tipificaciones.shape}  columns: {df_tipificaciones.columns.tolist()}')

In [ ]:
# LEFT JOIN → merged_df1
merged_df1 = pd.merge(
    df_principal,
    df_tipificaciones,
    how='left',
    left_on=['OpenchannelInteractionId', 'ContactId'],
    right_on=['id', 'ContactId'],
    suffixes=('', '_tip')
)

# Drop the redundant 'id' column brought in from tipificaciones
if 'id_tip' in merged_df1.columns:
    merged_df1 = merged_df1.drop(columns=['id_tip'])

print(f'merged_df1 shape : {merged_df1.shape}')
print(f'Columns          : {merged_df1.columns.tolist()}')
merged_df1.head(3)

## 2. Attach body_anonimizado

In [ ]:
df_anon = pd.read_excel(PATH_ANONIMIZADO)
print(f'df_anon shape  : {df_anon.shape}')
print(f'Columns        : {df_anon.columns.tolist()}')
df_anon.head(3)

In [ ]:
# Merge anonymized body into merged_df1 via 'id'
# If body_anonimizado is already in merged_df1, skip this block
if 'body_anonimizado' not in merged_df1.columns:
    merged_df1 = pd.merge(
        merged_df1,
        df_anon[['id', 'body_anonimizado']],
        on='id',
        how='left'
    )

# Drop rows without anonymized text
merged_df1 = merged_df1.dropna(subset=['body_anonimizado']).reset_index(drop=True)

print(f'Rows with body_anonimizado: {len(merged_df1)}')
merged_df1[['id', 'direction', 'OpenchannelInteractionId', 'ContactId', 'body_anonimizado']].head(5)

## 3. Preprocessing

In [ ]:
def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

merged_df1['text_clean'] = merged_df1['body_anonimizado'].apply(preprocess)

# Split by direction
df_in  = merged_df1[merged_df1['direction'] == 'in'].copy()
df_out = merged_df1[merged_df1['direction'] == 'out'].copy()

print(f"Messages 'in' (user) : {len(df_in)}")
print(f"Messages 'out' (agent): {len(df_out)}")

## 4. LDA Topic Modeling on body_anonimizado

We run LDA on **user messages** (`direction='in'`) — these carry the relevant clinical content.

In [ ]:
N_TOPICS = 6   # Adjust based on your domain knowledge

SPANISH_STOP_WORDS = [
    'de','la','el','en','y','a','los','del','se','las','por','un',
    'para','con','una','su','al','lo','como','más','pero','sus','le',
    'ya','o','este','si','porque','esta','entre','cuando','muy','sin',
    'sobre','también','me','hasta','hay','donde','quien','desde','todo',
    'nos','durante','todos','uno','les','ni','contra','otros','ese',
    'eso','ante','ellos','e','esto','mí','antes','algunos','qué','unos',
    'yo','otro','otras','otras','tanto','esa','estos','mucho','quienes',
    'nada','muchos','cual','poco','ella','estar','estas','mi','ha','vez',
    'hola','buenas','gracias','ok','vale','sí','no','que','te','es',
    'nombre'
]

vec = CountVectorizer(
    stop_words=SPANISH_STOP_WORDS,
    max_df=0.90,
    min_df=2,
    max_features=1000
)

texts_in = df_in['text_clean'].tolist()
dtm = vec.fit_transform(texts_in)
vocab = vec.get_feature_names_out()

print(f'Vocabulary size : {len(vocab)}')
print(f'DTM shape       : {dtm.shape}')

In [ ]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=30,
    learning_method='batch',
    random_state=42
)
lda.fit(dtm)

# Topic-probability matrix for all user messages
doc_topic_matrix = lda.transform(dtm)   # shape (n_messages, N_TOPICS)

print('\nTop 12 words per topic:')
for idx, topic in enumerate(lda.components_):
    top_words = [vocab[i] for i in topic.argsort()[-12:][::-1]]
    print(f'  Topic {idx}: {", ".join(top_words)}')

In [ ]:
# Attach topic probabilities and dominant topic to df_in
for t in range(N_TOPICS):
    df_in[f'topic_{t}'] = doc_topic_matrix[:, t]

df_in['dominant_topic'] = doc_topic_matrix.argmax(axis=1)

df_in[['id', 'body_anonimizado', 'dominant_topic'] +
      [f'topic_{t}' for t in range(N_TOPICS)]].head(8)

## 5. Conversation-Level Topic Aggregation

Group messages by `OpenchannelInteractionId` and compute the **mean topic distribution** per conversation.

In [ ]:
topic_cols = [f'topic_{t}' for t in range(N_TOPICS)]

conv_topics = (
    df_in
    .groupby('OpenchannelInteractionId')[topic_cols]
    .mean()
    .reset_index()
)
conv_topics['dominant_topic'] = conv_topics[topic_cols].values.argmax(axis=1)
conv_topics['n_messages'] = (
    df_in.groupby('OpenchannelInteractionId').size().values
)

print(f'Conversations: {len(conv_topics)}')
conv_topics.sort_values('n_messages', ascending=False).head(8)

## 6. BERT + LDA Ensemble — Risk Scoring

We use the same two-branch architecture as the suicide-risk ensemble notebook,
now applied directly to `body_anonimizado` messages.

In [ ]:
# ----------------------------------------------------------------
# BERT mean-pool embeddings
# Swap model for BETO if working exclusively in Spanish:
#   'dccuchile/bert-base-spanish-wwm-cased'
# ----------------------------------------------------------------
BERT_MODEL = 'bert-base-multilingual-cased'

print(f'Loading {BERT_MODEL} ...')
tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE)
bert_model.eval()
print('Done.')

In [ ]:
@torch.no_grad()
def bert_embeddings(texts: list[str], batch_size: int = 16) -> np.ndarray:
    """Return mean-pooled last-hidden-state embeddings."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(DEVICE)
        hidden = bert_model(**enc).last_hidden_state          # (B, L, 768)
        mask   = enc['attention_mask'].unsqueeze(-1).float()  # (B, L, 1)
        pooled = (hidden * mask).sum(1) / mask.sum(1)         # (B, 768)
        out.append(pooled.cpu().numpy())
    return np.vstack(out)

In [ ]:
print('Encoding messages with BERT (this may take a few minutes) ...')
X_bert = bert_embeddings(df_in['text_clean'].tolist())
print(f'BERT embeddings shape: {X_bert.shape}')  # (n_messages, 768)

In [ ]:
# LDA features (already computed in Section 4)
X_lda = doc_topic_matrix   # (n_messages, N_TOPICS)

# Concatenate
X_combined = np.hstack([X_bert, X_lda])   # (n_messages, 768 + N_TOPICS)

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

print(f'Combined feature matrix: {X_combined.shape}')

### 6a. Unsupervised risk proxy via BERT similarity

If **no labelled data** is available, we compute a **cosine similarity** between each
message embedding and a risk-keyword centroid as a proxy risk score.

> Replace this block with a trained classifier once you have annotated data.

In [ ]:
# Risk-seed phrases (Spanish)
RISK_SEEDS = [
    "no quiero seguir viviendo",
    "quiero hacerme daño",
    "no puedo más con mi vida",
    "pienso en suicidarme",
    "ya no tiene sentido vivir",
    "quiero que todo termine",
    "soy una carga para todos",
    "he pensado en quitarme la vida",
]

print('Encoding risk seed phrases ...')
seed_clean   = [preprocess(s) for s in RISK_SEEDS]
seed_emb     = bert_embeddings(seed_clean)             # (n_seeds, 768)
risk_centroid = seed_emb.mean(axis=0, keepdims=True)   # (1, 768)

# Cosine similarity between each message and the risk centroid
def cosine_sim(A, B):
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
    return (A_norm * B_norm).sum(axis=1)

risk_score = cosine_sim(X_bert, risk_centroid)  # (n_messages,)

# Normalise to [0, 1]
risk_score_norm = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min() + 1e-9)

df_in = df_in.copy()
df_in['bert_risk_score'] = risk_score_norm

print(f'Risk score — mean: {risk_score_norm.mean():.3f}  max: {risk_score_norm.max():.3f}')

### 6b. (Optional) Supervised meta-classifier — requires labelled data

In [ ]:
# -----------------------------------------------------------------------
# If you have a 'risk_label' column in merged_df1 (0/1), uncomment this:
# -----------------------------------------------------------------------

# y = df_in['risk_label'].values
# meta_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
# meta_clf.fit(X_scaled, y)
# df_in['ensemble_risk_prob'] = meta_clf.predict_proba(X_scaled)[:, 1]
# print(classification_report(y, meta_clf.predict(X_scaled),
#       target_names=['no_risk', 'suicide_risk']))

## 7. Conversation-Level Risk Aggregation

In [ ]:
conv_risk = (
    df_in
    .groupby('OpenchannelInteractionId')
    .agg(
        n_messages    = ('id', 'count'),
        mean_risk     = ('bert_risk_score', 'mean'),
        max_risk      = ('bert_risk_score', 'max'),
        dominant_topic= ('dominant_topic',  lambda x: x.mode()[0])
    )
    .reset_index()
    .sort_values('max_risk', ascending=False)
)

# Flag conversations above 75th-percentile max risk
threshold = conv_risk['max_risk'].quantile(0.75)
conv_risk['high_risk_flag'] = conv_risk['max_risk'] >= threshold

print(f'High-risk conversations (top 25%): {conv_risk["high_risk_flag"].sum()}')
conv_risk.head(10)

## 8. Visualizations

In [ ]:
# --- 8a. Top words per topic ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
colors = list(mcolors.TABLEAU_COLORS.values())

for idx, (ax, topic) in enumerate(zip(axes, lda.components_)):
    n = 12
    top_idx   = topic.argsort()[-n:][::-1]
    top_words = [vocab[i] for i in top_idx]
    top_vals  = topic[top_idx]
    top_vals  = top_vals / top_vals.sum()
    ax.barh(top_words[::-1], top_vals[::-1], color=colors[idx % len(colors)])
    ax.set_title(f'Topic {idx}', fontsize=11)
    ax.set_xlabel('Relative weight')

plt.suptitle('LDA — Top 12 Words per Topic (body_anonimizado)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('lda_top_words.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8b. Dominant topic distribution ---
topic_counts = df_in['dominant_topic'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(topic_counts.index, topic_counts.values,
       color=[colors[i % len(colors)] for i in topic_counts.index])
ax.set_xlabel('Topic')
ax.set_ylabel('Number of messages')
ax.set_title('Dominant Topic Distribution — user messages')
ax.set_xticks(range(N_TOPICS))
plt.tight_layout()
plt.savefig('dominant_topic_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8c. Risk score distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_in['bert_risk_score'], bins=30, color='tomato', edgecolor='white', alpha=0.85)
ax.axvline(threshold, color='black', linestyle='--', label=f'P75 threshold = {threshold:.2f}')
ax.set_xlabel('BERT risk score (normalised)')
ax.set_ylabel('Messages')
ax.set_title('Distribution of Risk Scores (body_anonimizado)')
ax.legend()
plt.tight_layout()
plt.savefig('risk_score_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8d. Heatmap — topic distribution per conversation (top 30 by message count) ---
top_convs = conv_topics.sort_values('n_messages', ascending=False).head(30)
heat_data = top_convs[topic_cols].values

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(heat_data, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(N_TOPICS))
ax.set_xticklabels([f'T{i}' for i in range(N_TOPICS)])
ax.set_yticks(range(len(top_convs)))
ax.set_yticklabels(top_convs['OpenchannelInteractionId'].astype(str), fontsize=7)
ax.set_xlabel('Topic')
ax.set_ylabel('Conversation ID')
ax.set_title('Topic Distribution — top 30 conversations')
plt.colorbar(im, ax=ax, label='Mean topic probability')
plt.tight_layout()
plt.savefig('conv_topic_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8e. Mean & max risk per conversation (top 40) ---
plot_df = conv_risk.head(40).sort_values('mean_risk', ascending=True)
y_pos   = range(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(y_pos, plot_df['max_risk'],  color='tomato',    alpha=0.6, label='Max risk')
ax.barh(y_pos, plot_df['mean_risk'], color='steelblue', alpha=0.8, label='Mean risk')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_df['OpenchannelInteractionId'].astype(str), fontsize=7)
ax.set_xlabel('Risk score')
ax.set_title('Mean & Max Risk per Conversation')
ax.legend()
plt.tight_layout()
plt.savefig('conv_risk_scores.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Export Results

In [ ]:
# Message-level output
msg_output_cols = [
    'id', 'OpenchannelInteractionId', 'ContactId',
    'direction', 'body_anonimizado',
    'dominant_topic', 'bert_risk_score'
] + topic_cols

df_in[msg_output_cols].to_excel('messages_topics_risk.xlsx', index=False)

# Conversation-level output (merge topics + risk)
conv_final = pd.merge(
    conv_topics,
    conv_risk[['OpenchannelInteractionId', 'mean_risk', 'max_risk', 'high_risk_flag']],
    on='OpenchannelInteractionId'
)
conv_final.to_excel('conversations_topics_risk.xlsx', index=False)

print('Exported:')
print('  messages_topics_risk.xlsx      — one row per message')
print('  conversations_topics_risk.xlsx — one row per conversation')

## 10. Inspect High-Risk Conversations

In [ ]:
high_risk_ids = conv_risk[conv_risk['high_risk_flag']]['OpenchannelInteractionId'].tolist()

print(f'High-risk conversation IDs: {high_risk_ids[:10]} ...')

# Show top-5 riskiest messages across all conversations
top_risk_msgs = (
    df_in[['OpenchannelInteractionId', 'body_anonimizado', 'dominant_topic', 'bert_risk_score']]
    .sort_values('bert_risk_score', ascending=False)
    .head(10)
)
pd.set_option('display.max_colwidth', 80)
top_risk_msgs